# Train RSSM World Model on MiniGrid

Training a Recurrent State-Space Model (RSSM) from DreamerV2 on MiniGrid-Empty-16x16-v0.

Based on the repository: https://github.com/mihalko711/tbank_intro_problem

## 1. Setup

Install dependencies and clone the repository.

In [ ]:
!pip install gymnasium minigrid matplotlib pyyaml Pillow tqdm -q

import sys
!git clone https://github.com/mihalko711/tbank_intro_problem.git
sys.path.insert(0, 'tbank_intro_problem')
%cd tbank_intro_problem

## 2. Imports, config, initialization

In [ ]:
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import clear_output
from tqdm.notebook import tqdm

from src import (
    RSSMWorldModel, collect_episode, evaluate,
    get_env_properties, make_minigrid_env, seed_everything,
)

with open('configs/minigrid_default.yml') as f:
    config = yaml.safe_load(f)
rssm_cfg = config['rssm']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
seed_everything(config['seed'])

env = make_minigrid_env(config['environment_name'], seed=config['seed'])
env_eval = make_minigrid_env(config['environment_name'], seed=config['seed'] + 999)
obs_shape, action_size = get_env_properties(env)
print(f'Obs shape: {obs_shape}, action size: {action_size}')

rssm = RSSMWorldModel(obs_shape, action_size, rssm_cfg, device)

## 3. Initial data collection

Collect episodes using a random policy to fill the replay buffer.

In [ ]:
print(f"Collecting {config['episodes_before_start']} initial episodes...")
for _ in range(config['episodes_before_start']):
    collect_episode(env, rssm, rssm.buffer)
print(f"Buffer size: {len(rssm.buffer)}")

## 4. Reconstruction visualization helper

Decodes latent states back to pixels and compares with ground truth observations.

In [ ]:
@torch.no_grad()
def visualize_reconstructions(rssm, buffer, num_samples=4):
    data = buffer.sample(num_samples, 2)
    obs = data['observations']

    obs_flat = obs[:, 1].to(rssm.device)
    encoded = rssm.encoder(obs_flat)

    recurrent = torch.zeros(num_samples, rssm.recurrent_size, device=rssm.device)
    latent, _ = rssm.posterior_net(torch.cat((recurrent, encoded), -1))
    full_state = torch.cat((recurrent, latent), -1)
    recon = rssm.decoder(full_state)

    obs_np = obs_flat.cpu().numpy()
    recon_np = recon.cpu().numpy()

    fig, axes = plt.subplots(2, num_samples, figsize=(4 * num_samples, 4))
    for i in range(num_samples):
        axes[0, i].imshow(np.transpose(obs_np[i], (1, 2, 0)))
        axes[0, i].set_title('Original')
        axes[0, i].axis('off')
        axes[1, i].imshow(np.transpose(recon_np[i].clip(0, 1), (1, 2, 0)))
        axes[1, i].set_title('Reconstruction')
        axes[1, i].axis('off')
    plt.tight_layout()
    plt.show()

## 5. Training loop

Alternates between gradient steps and environment interaction.
Shows live loss curves, periodic reconstructions, and evaluation rewards.

In [ ]:
iters = config['gradient_steps'] // config['replay_ratio']
loss_history = {'wm_loss': [], 'recon_loss': [], 'reward_loss': [], 'kl_loss': []}
eval_history = []

pbar = tqdm(range(iters), desc='Training')

for iteration in pbar:
    for _ in range(config['replay_ratio']):
        data = rssm.buffer.sample(rssm_cfg['batch_size'], rssm_cfg['batch_length'])
        _, metrics = rssm.train_step(data)
        rssm.total_gradient_steps += 1

    for k in loss_history:
        loss_history[k].append(metrics[k])

    collect_episode(env, rssm, rssm.buffer)

    if iteration % 10 == 0 and iteration > 0:
        clear_output(wait=True)
        fig, axes = plt.subplots(2, 3, figsize=(16, 8))
        colors = {'wm_loss': 'blue', 'recon_loss': 'green', 'reward_loss': 'orange', 'kl_loss': 'red'}
        for ax, (k, color) in zip(axes.flat[:4], colors.items()):
            ax.plot(loss_history[k], color=color)
            ax.set_title(k)
            ax.set_xlabel('Iteration')
        if eval_history:
            steps, means, stds = zip(*eval_history)
            axes.flat[4].errorbar(steps, means, yerr=stds, capsize=3)
            axes.flat[4].set_title('Eval reward')
            axes.flat[4].set_xlabel('Gradient step')
        axes.flat[5].text(0.5, 0.5, 'Reconstructions below \u2193',
                          ha='center', va='center', fontsize=14)
        axes.flat[5].axis('off')
        plt.tight_layout()
        plt.show()

        if iteration % 50 == 0:
            visualize_reconstructions(rssm, rssm.buffer, num_samples=4)

    gs = rssm.total_gradient_steps
    if config['save_checkpoints'] and gs % config['checkpoint_interval'] == 0:
        avg, std = evaluate(
            env_eval, rssm,
            lambda s: torch.nn.functional.one_hot(
                torch.randint(0, action_size, (1,), device=device), action_size
            ).float(),
            num_episodes=config['num_evaluation_episodes'],
        )
        eval_history.append((gs, avg, std))
        path = (
            f"{config['folder_names']['checkpoints_folder']}/"
            f"{config['environment_name']}_{config['run_name']}_{gs // 1000}k"
        )
        rssm.save_checkpoint(path)
        pbar.set_postfix(wm_loss=f"{metrics['wm_loss']:.2f}",
                         eval_reward=f"{avg:.2f}\u00b1{std:.2f}")

## 6. Save training metrics

In [ ]:
df_loss = pd.DataFrame(loss_history)
df_loss.to_csv('training_metrics.csv', index=False)
print(f'Loss history saved: {len(df_loss)} rows')

if eval_history:
    df_eval = pd.DataFrame(eval_history, columns=['gradient_step', 'mean_reward', 'std_reward'])
    df_eval.to_csv('eval_metrics.csv', index=False)
    print(f'Eval history saved: {len(df_eval)} rows')

final_path = (
    f"{config['folder_names']['checkpoints_folder']}/"
    f"{config['environment_name']}_{config['run_name']}_final"
)
rssm.save_checkpoint(final_path)
print(f'Final checkpoint saved: {final_path}.pth')

env.close()
env_eval.close()
print('Done.')